# Thesis Analysis: Data Logs
Comprehensive thesis analysis including NDCG, Router wins, Heatmaps, and Advanced Distributions.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import precision_recall_curve, auc

# --- BEAUTIFUL SEABORN STYLE ---
sns.set_theme(style="white", context="talk") # Clean white background
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'
%matplotlib inline


## 2. System Architecture Flow
Visual representation of the Hybrid Search query execution pipeline.

In [ ]:
try:
    import networkx as nx
    import matplotlib.patches as mpatches

    plt.figure(figsize=(10, 6))
    G = nx.DiGraph()
    G.add_node('User\nQuery', layer=0, color='#90EE90')
    G.add_node('FastAPI\nHandler', layer=1, color='#ADD8E6')
    G.add_node('Vector\nSearch', layer=2, color='#FFB6C1')
    G.add_node('Keyword\nSearch', layer=2, color='#FFB6C1')
    G.add_node('Hybrid\nFusion', layer=3, color='#D8BFD8')
    G.add_node('Final\nRanking', layer=4, color='#ADD8E6')
    G.add_node('AI Judge\n(Async)', layer=4, color='#FFD700')
    edges = [
        ('User\nQuery', 'FastAPI\nHandler'),
        ('FastAPI\nHandler', 'Vector\nSearch'),
        ('FastAPI\nHandler', 'Keyword\nSearch'),
        ('Vector\nSearch', 'Hybrid\nFusion'),
        ('Keyword\nSearch', 'Hybrid\nFusion'),
        ('Hybrid\nFusion', 'Final\nRanking'),
        ('Hybrid\nFusion', 'AI Judge\n(Async)')
    ]
    G.add_edges_from(edges)
    pos = nx.multipartite_layout(G, subset_key='layer')
    pos['Vector\nSearch'][1] += 0.05
    pos['Keyword\nSearch'][1] -= 0.05
    node_colors = [nx.get_node_attributes(G, 'color')[n] for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=2200, node_color=node_colors, edgecolors='black', linewidths=1)
    nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=15, edge_color='gray', width=1.5, connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='normal')
    plt.title("Architecture: Hybrid Search Pipeline", fontsize=12, fontweight='bold', pad=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("NetworkX/Matplotlib not fully available.")


In [ ]:
# Load Data
file_path = 'thesis_results_data_1766938498.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Loaded: {df.shape}")
except:
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"Loaded (latin1): {df.shape}")
    except Exception as e:
        print(f"Error: {e}")
        df = None

if df is not None and 'ai_relevant' in df.columns:
    df['ai_relevant_bool'] = df['ai_relevant'].astype(str).str.lower() == 'true'
    df['relevance_int'] = df['ai_relevant_bool'].astype(int)


In [ ]:
def calculate_ndcg(df, k=10):
    if df is None: return pd.DataFrame()
    results = []
    methods = {'Hybrid': 'hybrid_linear', 'RRF': 'rrf', 'Semantic': 'semantic_only', 'Keyword': 'keyword_refined'}
    if 'relevance_int' not in df.columns: return pd.DataFrame()
    for query, group in df.groupby('query'):
        for label, prefix in methods.items():
            rank_col = f"{prefix}_rank"
            if rank_col not in group.columns: continue
            sub = group.dropna(subset=[rank_col]).copy()
            sub['rank'] = pd.to_numeric(sub[rank_col], errors='coerce')
            sub = sub.sort_values('rank').head(k)
            dcg = 0
            for i, row in enumerate(sub.itertuples(), 1):
                dcg += (2**row.relevance_int - 1) / np.log2(i + 1)
            ideal = sub.sort_values('relevance_int', ascending=False)
            idcg = 0
            for i, row in enumerate(ideal.itertuples(), 1):
                idcg += (2**row.relevance_int - 1) / np.log2(i + 1)
            ndcg = dcg / idcg if idcg > 0 else 0
            results.append({'Query': query, 'Method': label, 'NDCG@10': ndcg})
    return pd.DataFrame(results)

def get_metrics(df):
    if df is None: return pd.DataFrame()
    data = []
    methods = {'Hybrid': 'hybrid_linear', 'RRF': 'rrf', 'Semantic': 'semantic_only', 'Keyword': 'keyword_refined'}
    for label, prefix in methods.items():
        if f"{prefix}_latency" in df.columns:
            lat = pd.to_numeric(df[f"{prefix}_latency"], errors='coerce').mean()
            data.append({'Method': label, 'Latency': lat})
    return pd.DataFrame(data)

def classify_query(q):
    q = str(q).lower().strip()
    if q.startswith('what'): return 'Fact (What)'
    if q.startswith('how'): return 'Process (How)'
    if q.startswith('explain'): return 'Concept'
    if ' vs ' in q or 'compare' in q: return 'Compare'
    return 'Other'


## 4. Performance Comparison: Aggregate Metrics (N=100 Queries)
Combined view of Quality (NDCG) and Efficiency (Latency).

In [ ]:
ndcg_df = calculate_ndcg(df)
latency_df = get_metrics(df)
final = pd.DataFrame()

if not ndcg_df.empty:
    metrics = ndcg_df.groupby('Method')['NDCG@10'].mean().reset_index()
    final = pd.merge(metrics, latency_df, on='Method')
else:
    final = latency_df
    final['NDCG@10'] = 0 

# --- PREP DATA ---
final['Method'] = final['Method'].replace({
    'Semantic': 'Semantic (Dense)',
    'RRF': 'Hybrid-RRF',
    'Hybrid': 'Hybrid-Linear',
    'Keyword': 'Lexical (ts_rank)'
})
final['Efficiency (NDCG/ms)'] = final['NDCG@10'] / final['Latency']
final = final.rename(columns={'NDCG@10': 'NDCG@10 (Mean)', 'Latency': 'Latency (Mean ms)'})
row_order = ['Semantic (Dense)', 'Hybrid-RRF', 'Hybrid-Linear', 'Lexical (ts_rank)']
final['Method'] = pd.Categorical(final['Method'], categories=row_order, ordered=True)
final = final.sort_values('Method')

# Display Table
print("--- AGGREGATE PERFORMANCE TABLE (N=100) ---")
from IPython.display import display
styled_df = final.style.format({
    'NDCG@10 (Mean)': '{:.3f}',
    'Latency (Mean ms)': '{:.0f}',
    'Efficiency (NDCG/ms)': '{:.2e}'
}).hide(axis='index')
display(styled_df)

# --- SCATTER PLOT (Quality vs Latency) --- [ADDED THIS SECTION]
print("\n--- QUALITY vs LATENCY TRADE-OFF ---")
if not final.empty and 'Latency (Mean ms)' in final.columns:
    plt.figure(figsize=(9, 6))
    
    # Scatter Plot
    ax = sns.scatterplot(data=final, x='Latency (Mean ms)', y='NDCG@10 (Mean)', 
                         hue='Method', s=300, palette='viridis', legend=False)
    
    # Axis Formatting
    plt.title('Performance Trade-off: Quality vs Latency', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Avg Latency (ms) [Lower is Better]', fontsize=12, fontweight='bold')
    plt.ylabel('NDCG@10 [Higher is Better]', fontsize=12, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Labels next to dots
    for i, row in final.iterrows():
        plt.text(row['Latency (Mean ms)'] + 15, row['NDCG@10 (Mean)'] + 0.002, 
                 row['Method'], fontsize=12, fontweight='bold', 
                 bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    
    sns.despine()
    plt.show()

# --- DUAL AXIS PLOT (Original) ---
if not final.empty and 'Latency (Mean ms)' in final.columns:
    fig, ax1 = plt.subplots(figsize=(11, 7))
    bar_color = "#4c72b0" 
    line_color = "#c44e52" 
    sns.barplot(data=final, x='Method', y='NDCG@10 (Mean)', ax=ax1, color=bar_color, alpha=0.9)
    ax1.set_ylabel('Ranking Quality (NDCG@10)', color=bar_color, fontsize=14, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor=bar_color, labelsize=12)
    ax1.tick_params(axis='x', labelsize=11)
    ax1.set_ylim(0, 1.25) 
    ax1.set_xlabel('') 
    sns.despine(ax=ax1, top=True, right=False)
    for p in ax1.patches:
        ax1.annotate(f'{p.get_height():.3f}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 10), 
                     textcoords='offset points', fontsize=12, fontweight='bold', color=bar_color)
    ax2 = ax1.twinx()
    sns.lineplot(data=final, x='Method', y='Latency (Mean ms)', ax=ax2, 
                 color=line_color, marker='o', linewidth=3, markersize=10)
    ax2.set_ylabel('Latency (ms)', color=line_color, fontsize=14, fontweight='bold', rotation=-90, labelpad=20)
    ax2.tick_params(axis='y', labelcolor=line_color, labelsize=12)
    ax2.set_ylim(0, final['Latency (Mean ms)'].max() * 1.35)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['right'].set_color(line_color)
    ax2.spines['right'].set_linewidth(2)
    ax1.spines['left'].set_color(bar_color)
    ax1.spines['left'].set_linewidth(2)
    ax1.spines['bottom'].set_linewidth(1.5)
    for i, row in final.iterrows():
        ax2.text(i, row['Latency (Mean ms)'] - 40, f"{row['Latency (Mean ms)']:.0f}ms", 
                 color=line_color, fontweight='bold', ha='center', fontsize=11,
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=0.5))
    plt.title('Trade-off Analysis: Relevancy vs Latency', fontsize=16, fontweight='bold', pad=20)
    plt.show()


### 4.4 Category Performance (Heatmap & Table)

In [ ]:
if not ndcg_df.empty:
    ndcg_df['Type'] = ndcg_df['Query'].apply(classify_query)
    ht_data = ndcg_df.groupby(['Method', 'Type'])['NDCG@10'].mean().unstack()
    print("Table: NDCG@10 Per Category")
    display(ht_data.style.format("{:.1%}") if hasattr(ht_data, 'style') else ht_data)
    plt.figure(figsize=(9, 5))
    ax = sns.heatmap(ht_data, annot=True, fmt='.1%', cmap='RdYlGn', linewidths=1.5, annot_kws={"size": 12})
    plt.title('Expected Quality by Query Type', fontweight='bold', fontsize=14, pad=15)
    plt.xlabel('Query Category', fontsize=12, fontweight='bold')
    plt.ylabel('Search Method', fontsize=12, fontweight='bold')
    plt.xticks(fontsize=12, rotation=0)
    plt.yticks(fontsize=12, rotation=0)
    plt.tight_layout()
    plt.show()


## 5. Router Analysis (Winner Distribution)
Frequency of each algorithm providing the best result, broken down by query category.

In [ ]:
if not ndcg_df.empty:
    piv = ndcg_df.pivot(index='Query', columns='Method', values='NDCG@10')
    piv['Winner'] = piv.idxmax(axis=1)
    piv['Category'] = piv.index.map(classify_query)
    
    # Prepare Data
    win_counts = piv['Winner'].value_counts().reset_index()
    win_counts.columns = ['Method', 'Count']
    win_counts['Percentage'] = (win_counts['Count'] / win_counts['Count'].sum() * 100).round(1)

    # Chart
    plt.figure(figsize=(8, 4)) 
    ax = sns.barplot(data=win_counts, y='Method', x='Count', palette='coolwarm')
    sns.despine(right=True, top=True)
    
    # Labels
    for i, p in enumerate(ax.patches):
        width = p.get_width()
        count = int(width) if not np.isnan(width) else 0
        pct = win_counts.iloc[i]['Percentage'] if i < len(win_counts) else 0
        ax.annotate(f'{count} ({pct}%)',
                    (width, p.get_y() + p.get_height() / 2),
                    xytext=(5, 0), textcoords='offset points',
                    va='center', fontsize=11, fontweight='bold', color='black')

    plt.title('Distribution of Winning Algorithms', fontweight='bold', fontsize=14, pad=15)
    plt.xlabel('Number of Wins', fontsize=12, fontweight='bold')
    plt.ylabel('', fontsize=12)
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=12)
    plt.show()

    # TABLE REMAINS
    print("\n--- TABLE: ALGORITHM WINS BY CATEGORY ---")
    win_table = pd.crosstab(piv['Category'], piv['Winner'])
    win_table['Total Wins'] = win_table.sum(axis=1)
    display(win_table.style.background_gradient(cmap='Blues', subset=win_table.columns[:-1]))
    
    print("\n=== WINNER QUALITATIVE ANALYSIS ===")
    for method in piv['Winner'].unique():
        count = len(piv[piv['Winner'] == method])
        print(f"\n🏆 {method.upper()} ({count} wins)")
        examples = piv[piv['Winner'] == method].index.tolist()[:3]
        for q in examples:
            print(f"   - {q} (NDCG: {piv.loc[q, method]:.2f})")


## 6. Advanced Signal Analysis

In [ ]:
if 'relevance_int' in df.columns:
    plt.figure(figsize=(7, 5))
    methods = {'Hybrid': 'hybrid_linear', 'RRF': 'rrf', 'Semantic': 'semantic_only', 'Keyword': 'keyword_refined'}
    for label, prefix in methods.items():
        score_col = f"{prefix}_score"
        if score_col in df.columns:
            subset = df.dropna(subset=[score_col, 'relevance_int'])
            if len(subset) > 0:
                y_true = subset['relevance_int']
                y_scores = subset[score_col]
                precision, recall, _ = precision_recall_curve(y_true, y_scores)
                pr_auc = auc(recall, precision)
                plt.plot(recall, precision, label=f'{label} (AUC={pr_auc:.2f})')
    plt.title('Precision-Recall Curves', fontweight='bold', fontsize=12)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend()
    plt.show()
    corr_data = pd.DataFrame()
    for label, prefix in methods.items():
        if f"{prefix}_score" in df.columns:
            corr_data[label] = pd.to_numeric(df[f"{prefix}_score"], errors='coerce')
    if not corr_data.empty:
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr_data.corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Score Correlation Between Methods', fontweight='bold', fontsize=12)
        plt.show()
